In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import plotly.express as px
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

# Check if GPU is available (will use CPU if not)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

df = pd.read_csv("../data/validated/ipo_validated.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print("Data loaded!")

Using device: cpu
Shape: (259, 20)
Data loaded!


In [2]:
# LSTM needs sequences — we use a sliding window of 10 IPOs
# to predict the 11th IPO's listing gain
# This captures market momentum — if last 10 IPOs did well,
# the next one might too

SEQUENCE_LENGTH = 10

FEATURES = [
    "issue_size_cr",
    "qib_subscription",
    "hni_subscription",
    "rii_subscription",
    "total_subscription",
    "issue_price",
    "nifty_30d_return",
    "Month",
    "Quarter"
]

TARGET = "listing_gains_pct"

# Fill any nulls
df[FEATURES] = df[FEATURES].fillna(0)

# Scale features to 0-1 range — LSTM trains better on scaled data
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[FEATURES])
y_scaled = scaler_y.fit_transform(df[[TARGET]])

print(f"Features scaled: {X_scaled.shape}")

# Build sequences
def build_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = build_sequences(X_scaled, y_scaled, SEQUENCE_LENGTH)

print(f"Sequences built!")
print(f"X shape: {X_seq.shape}  → (samples, timesteps, features)")
print(f"y shape: {y_seq.shape}  → (samples, 1)")

Features scaled: (259, 9)
Sequences built!
X shape: (249, 10, 9)  → (samples, timesteps, features)
y shape: (249, 1)  → (samples, 1)


In [3]:
# 80% train, 20% test — keep time order (no shuffle!)
split = int(len(X_seq) * 0.8)

X_train = torch.FloatTensor(X_seq[:split]).to(device)
X_test  = torch.FloatTensor(X_seq[split:]).to(device)
y_train = torch.FloatTensor(y_seq[:split]).to(device)
y_test  = torch.FloatTensor(y_seq[split:]).to(device)

print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples : {X_test.shape[0]}")

Train samples: 199
Test samples : 50


In [4]:
class IPO_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(IPO_LSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Take only the last timestep output
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out)

# Initialize model
model = IPO_LSTM(
    input_size  = len(FEATURES),
    hidden_size = 64,
    num_layers  = 2,
    dropout     = 0.2
).to(device)

print("LSTM Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

LSTM Model Architecture:
IPO_LSTM(
  (lstm): LSTM(9, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 54593


In [5]:
EPOCHS    = 100
LR        = 0.001
BATCH     = 16

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=30, gamma=0.5
)

train_losses = []
test_losses  = []

print("Training LSTM...")
print("-" * 40)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    pred   = model(X_train)
    loss   = criterion(pred, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())

    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test)
        test_loss = criterion(test_pred, y_test)
        test_losses.append(test_loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {loss.item():.4f} | "
              f"Test Loss: {test_loss.item():.4f}")

print("\nTraining complete!")

Training LSTM...
----------------------------------------
Epoch  20/100 | Train Loss: 0.1412 | Test Loss: 0.2286
Epoch  40/100 | Train Loss: 0.0572 | Test Loss: 0.0675
Epoch  60/100 | Train Loss: 0.0514 | Test Loss: 0.0606
Epoch  80/100 | Train Loss: 0.0419 | Test Loss: 0.0585
Epoch 100/100 | Train Loss: 0.0514 | Test Loss: 0.0583

Training complete!


In [6]:
fig = px.line(
    y=[train_losses, test_losses],
    labels={"x": "Epoch", "y": "Loss"},
    title="LSTM Training vs Test Loss"
)
fig.data[0].name = "Train Loss"
fig.data[1].name = "Test Loss"
fig.update_layout(legend_title="Loss Type")
fig.show()

In [7]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test).cpu().numpy()
    y_test_np     = y_test.cpu().numpy()

# Inverse transform back to original scale
y_pred_actual = scaler_y.inverse_transform(y_pred_scaled)
y_test_actual = scaler_y.inverse_transform(y_test_np)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2  = r2_score(y_test_actual, y_pred_actual)

print("=== LSTM Results ===")
print(f"MAE : {mae:.2f}%")
print(f"R2  : {r2:.4f}")

# Plot actual vs predicted
fig = px.line(
    title="LSTM — Actual vs Predicted IPO Listing Gains"
)
fig.add_scatter(
    y=y_test_actual.flatten(),
    mode="lines",
    name="Actual",
    line=dict(color="#1D9E75")
)
fig.add_scatter(
    y=y_pred_actual.flatten(),
    mode="lines",
    name="Predicted",
    line=dict(color="#FAC775", dash="dash")
)
fig.update_layout(
    xaxis_title="IPO Index",
    yaxis_title="Listing Gain (%)"
)
fig.show()

=== LSTM Results ===
MAE : 44.95%
R2  : -1.0333


In [8]:
os.makedirs("../src", exist_ok=True)

# Save model weights
torch.save(model.state_dict(), "../src/lstm_model.pth")

# Save scalers — needed for predictions in the app
with open("../src/scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)

with open("../src/scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)

with open("../src/lstm_features.pkl", "wb") as f:
    pickle.dump(FEATURES, f)

print("LSTM model and scalers saved!")
print("  src/lstm_model.pth")
print("  src/scaler_X.pkl")
print("  src/scaler_y.pkl")
print("  src/lstm_features.pkl")

LSTM model and scalers saved!
  src/lstm_model.pth
  src/scaler_X.pkl
  src/scaler_y.pkl
  src/lstm_features.pkl
